In [ ]:
import csv
import pandas as pd
import matplotlib.pyplot as plt
import fasttext
import mlflow

from mlflow.tracking import MlflowClient
from sklearn.model_selection import train_test_split
from gensim.utils import simple_preprocess

import numpy as np
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report

In [ ]:
MLFLOW_TRACKING_URI = "http://localhost:2000"
MLFLOW_EXPERIMENT_NAME = "fasttext_training"
MLFLOW_RUN_NAME = "nplus1 classification"

MLFLOW_PREPROCESS_RUN_ID = "e48ec02695424bedaab895917a6b19f1"
MLFLOW_DF_ARTIFACT_RUN_ID = "e48ec02695424bedaab895917a6b19f1"
MLFLOW_DF_ARTIFACT_PATH = "data/multigenres_df.csv"

RANDOM_STATE = 67

params = {"lr": 0.1, "epoch": 150, "wordNgrams": 1, "loss": "ova"}


In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
mlflow.start_run(run_name=MLFLOW_RUN_NAME)
mlflow.set_tag("upstream.preprocess_run_id", MLFLOW_PREPROCESS_RUN_ID)
mlflow.log_params(params)

client = MlflowClient()

In [ ]:
local_path = client.download_artifacts(run_id=MLFLOW_DF_ARTIFACT_RUN_ID, path=MLFLOW_DF_ARTIFACT_PATH)

df = pd.read_csv(local_path)

column_names = ['Label', 'Text', 'source_id', 'source_type']
df.columns = column_names
df = df[df['source_type'] == 'nplus1']
df.drop(columns=['source_id', 'source_type'], inplace=True)
df.head()

In [ ]:
df['Label'] = df['Label'].apply(lambda x: x.replace('\"', ''))

In [ ]:
train, test = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE)

In [ ]:
train.Label.unique()

In [ ]:
test.Label.unique()

In [ ]:
len(train)

In [ ]:
len(test)

In [ ]:
train.iloc[:, 1] = train.iloc[:, 1].apply(lambda x: ' '.join(simple_preprocess(str(x))))
test.iloc[:, 1] = test.iloc[:, 1].apply(lambda x: ' '.join(simple_preprocess(str(x))))

train.iloc[:, 0] = train.iloc[:, 0].apply(lambda x: '__label__' + str(x))
test.iloc[:, 0] = test.iloc[:, 0].apply(lambda x: '__label__' + str(x))

In [ ]:
train[['Label', 'Text']].to_csv('train_total.txt',
                                          index = False,
                                          sep = '\t',
                                          header = None,
                                     quoting = csv.QUOTE_NONE,
                                          escapechar = " "
                                )

test[['Label', 'Text']].to_csv('test_total.txt',
                                     index = False,
                                     sep = '\t',
                                     header = None,
                                     quoting = csv.QUOTE_NONE,
                                          escapechar = " "
                               )

In [ ]:
model = fasttext.train_supervised('train_total.txt', **params)

In [ ]:
acc, precision, recall = model.test('test_total.txt')
mlflow.log_metric('accuracy', acc)
mlflow.log_metric('recall', recall)
mlflow.log_metric('precision', precision)

In [ ]:
model.test('test_total.txt')

In [ ]:
def get_fasttext_predictions(model, test_file):
    y_true, y_pred = [], []

    with open(test_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split('\t', 1)

            true_label = parts[0].replace('__label__', '')
            text = parts[1]

            pred_label, _ = model.predict(text)
            pred_label = pred_label[0].replace('__label__', '')

            y_true.append(true_label)
            y_pred.append(pred_label)

    return y_true, y_pred

In [ ]:
def log_classification_report(model, test_file):
    y_true, y_pred = get_fasttext_predictions(model, test_file)

    report_dict = classification_report(
        y_true, y_pred,
        output_dict=True,
        zero_division=0
    )
    report_str = classification_report(y_true, y_pred, zero_division=0)

    print(report_str)

    mlflow.log_metrics({
        "accuracy":          report_dict.get('accuracy', 0),
        "macro_f1":          report_dict['macro avg']['f1-score'],
        "weighted_f1":       report_dict['weighted avg']['f1-score'],
        "macro_precision":   report_dict['macro avg']['precision'],
        "macro_recall":      report_dict['macro avg']['recall'],
    })

    mlflow.log_dict(report_dict, 'data/report_dict.json')

    return report_dict

In [ ]:
log_classification_report(model, "test_total.txt")

In [ ]:
def get_embeddings_for_viz(embeddings, authors, max_per_author: int = 50):
    """
    Берём по max_per_author эмбеддингов от каждого автора.
    Ограничиваем чтобы график не был перегружен.
    """
    from collections import defaultdict

    author_indices = defaultdict(list)
    for i, author in enumerate(authors):
        author_indices[author].append(i)

    selected_indices = []
    selected_authors = []

    for author, indices in author_indices.items():
        chosen = indices[:max_per_author]
        selected_indices.extend(chosen)
        selected_authors.extend([author] * len(chosen))

    matrix = np.stack(embeddings)[selected_indices]
    return matrix, selected_authors


def reduce_2d(matrix: np.ndarray, method: str = 'tsne') -> np.ndarray:
    """
    Понижение размерности до 2D.

    Сначала PCA до 50 измерений — ускоряет t-SNE без потери структуры.
    Затем t-SNE или UMAP до 2D.

    method: 'tsne' или 'umap'
    """
    # PCA — предварительное сжатие
    n_pca = min(50, matrix.shape[1], matrix.shape[0] - 1)
    pca   = PCA(n_components=n_pca, random_state=42)
    reduced = pca.fit_transform(matrix)

    if method == 'tsne':
        tsne = TSNE(
            n_components=2,
            perplexity=min(30, len(matrix) // 5),
            random_state=42,
            n_iter=1000,
            metric='cosine'   # эмбеддинги нормализованы — косинус уместен
        )
        return tsne.fit_transform(reduced)

    elif method == 'umap':
        import umap
        reducer = umap.UMAP(
            n_components=2,
            n_neighbors=15,
            min_dist=0.1,
            metric='cosine',
            random_state=42
        )
        return reducer.fit_transform(reduced)


def reduce_3d(matrix: np.ndarray) -> np.ndarray:
    """Понижение до 3D через UMAP — t-SNE в 3D работает хуже."""
    import umap
    reducer = umap.UMAP(
        n_components=3,
        n_neighbors=15,
        min_dist=0.1,
        metric='cosine',
        random_state=42
    )
    n_pca   = min(50, matrix.shape[1], matrix.shape[0] - 1)
    reduced = PCA(n_components=n_pca, random_state=42).fit_transform(matrix)
    return reducer.fit_transform(reduced)


# ── 2D график ──────────────────────────────────────────────────────

def plot_2d(embeddings, authors,
            method: str = 'tsne',
            max_per_author: int = 50,
            figsize: tuple = (14, 10)):

    matrix, authors = get_embeddings_for_viz(embeddings, authors, max_per_author)
    coords          = reduce_2d(matrix, method=method)
    unique_authors  = sorted(set(authors))

    # Цвета — tab20 для до 20 авторов, иначе hsv
    cmap   = plt.cm.get_cmap('tab20' if len(unique_authors) <= 20 else 'hsv',
                              len(unique_authors))
    colors = {a: cmap(i) for i, a in enumerate(unique_authors)}

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor('#0f0f1a')
    fig.patch.set_facecolor('#0f0f1a')

    for author in unique_authors:
        mask = [a == author for a in authors]
        xs   = coords[mask, 0]
        ys   = coords[mask, 1]
        ax.scatter(xs, ys,
                   c=[colors[author]],
                   label=author,
                   alpha=0.7,
                   s=25,
                   edgecolors='none')

    ax.set_title(f'Эмбеддинги авторов ({method.upper()})',
                 color='white', fontsize=14, pad=15)
    ax.tick_params(colors='gray')
    for spine in ax.spines.values():
        spine.set_color('#333355')

    ax.legend(
        bbox_to_anchor=(1.02, 1), loc='upper left',
        fontsize=7, ncol=max(1, len(unique_authors) // 25),
        facecolor='#1a1a2e', edgecolor='#333355', labelcolor='white'
    )

    plt.tight_layout()
    plt.savefig(f'embeddings_2d_{method}.png', dpi=150,
                bbox_inches='tight', facecolor='#0f0f1a')
    #plt.show()
    print(f"Сохранено: embeddings_2d_{method}.png")


In [ ]:
store = []
for row in train.iterrows():
    store.append(
        {
            'author': row[1]['Label'].replace('__label__', ''),
            'text': model.get_sentence_vector(row[1]['Text']),
        }
    )

In [ ]:
authors = [x['author'] for x in store]
embeddings = [x['text'] for x in store]

In [ ]:
plot_2d(embeddings, authors, 'umap')

In [ ]:
mlflow.log_artifact('embeddings_2d_umap.png', 'plots/embeddings_2d_train_umap.png')

In [ ]:
store = []
for row in test.iterrows():
    store.append(
        {
            'author': row[1]['Label'].replace('__label__', ''),
            'text': model.get_sentence_vector(row[1]['Text']),
        }
    )

In [ ]:
authors = [x['author'] for x in store]
embeddings = [x['text'] for x in store]

In [ ]:
plot_2d(embeddings, authors, 'umap')

In [ ]:
mlflow.log_artifact('embeddings_2d_umap.png', 'plots/embeddings_2d_umap_test.png')

In [ ]:
len(authors)

In [ ]:
#mlflow.log_artifact('model.bin', 'models/model.bin')
mlflow.end_run()